In [ ]:
#@title Cell 1 - Notebook overview

from IPython.display import display, Markdown

display(Markdown(r"""
# Simulation 06: Weakly supported virtual pathogen–plasmid combinations

## Purpose

Simulation 6 starts from the finalized Simulation 1 biological model and changes only the
pattern of observed pathogen–plasmid combinations.

The central question is:

\[
\boxed{\text{Does recovery become less accurate when a virtual pathogen–plasmid target is poorly supported by similar observed combinations?}}
\]

The model remains

\[
y_{ij}
=
\alpha
+
\boldsymbol{\beta}_C^T\mathbf c_i
+
\boldsymbol{\beta}_P^T\mathbf p_j
+
\mathbf c_i^T\mathbf B\mathbf p_j
+
u_i
+
\varepsilon_{ij}.
\]

The recovery targets remain

\[
\Delta_{ij}=y_{ij}-y_{i0}
\]

and

\[
\Delta\Delta_{ik,j}=\Delta_{ij}-\Delta_{kj}.
\]

## What is unchanged from Simulation 1

- 200 pathogen chromosomal backgrounds.
- 20 blaTEM-1 plasmids plus plasmid-free state \(P_0\).
- 30 AMR-related genes and their paired 300-bp putative promoter regions: 60 chromosome features.
- The same chromosomal state probabilities and biological effect sizes.
- The same empirical promoter-genotype/copy-number sampling.
- The same five plasmid features.
- The same sparse chromosome–plasmid interaction structure.
- 500 additional biallelic SNPs used only to construct \(K\).
- The same \(\alpha\), \(\sigma_g^2\), and \(\sigma_e^2\).
- The same REML/GLS fitting framework.

## Simulation 6 observation pattern

Of the 4,000 P1–P20 pathogen–plasmid combinations:

- 600 are withheld in local clusters around 10 pathogen–plasmid centres;
- 600 additional combinations are withheld randomly;
- all 200 P0 states remain observed.

Thus 1,200 of 4,000 P+ combinations are withheld and 2,800 remain observed.

The observed design must retain every pathogen, every plasmid, and full fixed-effect rank 366.

## Local support

For a withheld target \((i,j)\), distance is calculated only to observed P+ combinations.

The chromosome block contains the 60 AMR-related features
(30 genes + 30 paired 300-bp putative promoter regions).
The 500 background SNPs used for \(K\) are not included in this distance.

For an observed combination \((k,l)\),

\[
d((i,j),(k,l))
=
\sqrt{
\frac{1}{60}\sum_{r=1}^{60}
\left(\frac{c_{ir}-c_{kr}}{s_{C,r}}\right)^2
+
\frac{1}{5}\sum_{q=1}^{5}
\left(\frac{p_{jq}-p_{lq}}{s_{P,q}}\right)^2
}.
\]

The support score \(d_{ij}\) is the mean distance to the five nearest observed
P+ combinations.

Among the 1,200 withheld targets:

- lowest 25% of \(d_{ij}\): well supported;
- highest 25% of \(d_{ij}\): poorly supported;
- middle 50%: not used in the primary comparison.

For \(\Delta\Delta_{ik,j}\), each withheld target \((i,j)\) is compared with pathogens
\(k\) for which the same plasmid \(j\) was actually observed.

## Primary criterion

Compare RMSE in the well-supported and poorly-supported target groups.

A 10% increase is the pre-specified starting point for a meaningful loss:

\[
\frac{\mathrm{RMSE}_{poor}}{\mathrm{RMSE}_{well}}>1.10.
\]

The same direction should also be seen consistently across the 100 simulation replicates.
"""))

print("Cell 1: PASS")


In [ ]:
#@title Cell 2 - Load public plasmid-feature input and define fixed settings

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import cdist

REPO_ROOT = Path.cwd()
DATA_FILE = REPO_ROOT / "data" / "plasmid_feature_pairs.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / "simulation_06_weak_local_support"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Public plasmid-feature input was not found:\n"
        f"{DATA_FILE}\n"
        "Run the notebook from the repository root."
    )

empirical_pairs = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

KEY_SITE_COLUMNS = [
    "sutcliffe_32_nt",
    "sutcliffe_162_nt",
    "sutcliffe_175_nt",
]

required_columns = [
    *KEY_SITE_COLUMNS,
    "CN_TEM1",
]

missing_columns = [
    c for c in required_columns
    if c not in empirical_pairs.columns
]

if missing_columns:
    raise RuntimeError(
        "The public plasmid-feature input is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

empirical_pairs = empirical_pairs[
    required_columns
].copy()

for col in KEY_SITE_COLUMNS:
    empirical_pairs[col] = (
        empirical_pairs[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

empirical_pairs["CN_TEM1"] = pd.to_numeric(
    empirical_pairs["CN_TEM1"],
    errors="coerce",
)

valid_nt = {"A", "C", "G", "T"}

complete_mask = (
    empirical_pairs[KEY_SITE_COLUMNS]
    .apply(lambda s: s.isin(valid_nt))
    .all(axis=1)
    & empirical_pairs["CN_TEM1"].notna()
    & (empirical_pairs["CN_TEM1"] > 0)
)

empirical_pairs = (
    empirical_pairs.loc[complete_mask]
    .reset_index(drop=True)
)

if len(empirical_pairs) < 20:
    raise RuntimeError(
        f"Only {len(empirical_pairs)} complete promoter/copy-number pairs remain; "
        "at least 20 are required."
    )

CN_REFERENCE = float(
    empirical_pairs["CN_TEM1"].median()
)

if not np.isfinite(CN_REFERENCE) or CN_REFERENCE <= 0:
    raise RuntimeError(
        "The median CN_TEM1 is not positive."
    )

# -------------------------------------------------------------------------
# Simulation 1 biological settings: unchanged
# -------------------------------------------------------------------------

MASTER_SEED = 20260906

N_PATHOGENS = 200
N_PLASMIDS = 20

PREDEFINED_GENES = [
    "acrB", "acrR", "ampC", "basR", "cirA", "cyaA", "fabI", "folP", "ftsI",
    "gyrA", "marR", "nfsA", "nfsB", "ompC", "ompF", "parC", "parE", "pmrB",
    "ptsI", "rpoB", "rpsL", "soxR", "soxS", "uhpT", "acrA", "tolC", "marA",
    "rob", "ompR", "envZ",
]

TARGET_FEATURE_LABELS = []
for gene in PREDEFINED_GENES:
    TARGET_FEATURE_LABELS.extend([
        f"{gene}:coding",
        f"{gene}:upstream_300bp",
    ])

D_C = len(TARGET_FEATURE_LABELS)

PLASMID_FEATURE_LABELS = [
    "TEM1_plasmid_presence",
    "C32T",
    "G162T",
    "G175A",
    "log2_CN_relative_to_empirical_median",
]
D_P = len(PLASMID_FEATURE_LABELS)

CHROMOSOMAL_STATES = np.array([-1.0, 0.0, 1.0])
CHROMOSOMAL_STATE_PROBS = np.array([0.15, 0.70, 0.15])

N_BACKGROUND_SNPS = 500
ALLELE_FREQ_LOW = 0.10
ALLELE_FREQ_HIGH = 0.90

ALPHA_TRUE = -2.5

SIGMA_G_TRUE = 0.40
SIGMA_E_TRUE = 0.32
SIGMA_G2_TRUE = SIGMA_G_TRUE ** 2
SIGMA_E2_TRUE = SIGMA_E_TRUE ** 2

BETA_P_TRUE = np.array([
    0.50,
    0.50,
    0.25,
    0.00,
    0.25,
], dtype=float)

INTERACTION_MAGNITUDE = 0.10

# -------------------------------------------------------------------------
# Simulation 6 change
# -------------------------------------------------------------------------

MISSING_PPLUS_FRACTION = 0.30

N_CLUSTER_CENTRES = 10
N_CLUSTERED_WITHHELD = 600
N_RANDOM_WITHHELD = 600

N_SUPPORT_NEIGHBOURS = 5
SUPPORT_QUARTILE_FRACTION = 0.25

GO_RMSE_RATIO = 1.10

if (
    N_CLUSTERED_WITHHELD
    + N_RANDOM_WITHHELD
    != int(
        round(
            MISSING_PPLUS_FRACTION
            * N_PATHOGENS
            * N_PLASMIDS
        )
    )
):
    raise RuntimeError(
        "Clustered + random withheld counts do not equal the requested "
        "30% missing P+ total."
    )

N_SIM_REPLICATES = 100
N_BOOTSTRAP = 200

RUN_FULL_BOOTSTRAP_COVERAGE = False

print("=" * 90)
print("SIMULATION 06 — FIXED SETTINGS")
print("=" * 90)
print(f"Empirical promoter/CN pairs available: {len(empirical_pairs):,}")
print(f"Empirical median CN_TEM1:               {CN_REFERENCE:.6f}")
print(f"Pathogens:                              {N_PATHOGENS}")
print(f"Plasmids:                               {N_PLASMIDS}")
print(f"Chromosome support features:            {D_C} = 30 genes + 30 upstream regions")
print(f"Plasmid support features:               {D_P}")
print(f"Complete observations before masking:   {N_PATHOGENS * (N_PLASMIDS + 1):,}")
print(f"Cluster centres:                        {N_CLUSTER_CENTRES}")
print(f"Clustered P+ combinations withheld:     {N_CLUSTERED_WITHHELD}")
print(f"Random P+ combinations withheld:        {N_RANDOM_WITHHELD}")
print(f"Total P+ missing fraction:              {MISSING_PPLUS_FRACTION:.0%}")
print(f"Nearest neighbours for support:         {N_SUPPORT_NEIGHBOURS}")
print(f"Primary support quartiles:              lowest/highest {SUPPORT_QUARTILE_FRACTION:.0%}")
print(f"Starting GO RMSE ratio:                 > {GO_RMSE_RATIO:.2f}")
print("P0 observations retained:               100%")
print(f"alpha:                                  {ALPHA_TRUE}")
print(f"sigma_g:                                {SIGMA_G_TRUE}")
print(f"sigma_e:                                {SIGMA_E_TRUE}")
print(f"Output directory:                       {OUTPUT_DIR}")
print("\nCell 2: PASS")


In [ ]:
#@title Cell 3 - Define the Simulation 1 biological model and Simulation 6 support design

def feature_index(gene, region):
    label = (
        f"{gene}:coding"
        if region == "coding"
        else f"{gene}:upstream_300bp"
    )
    return TARGET_FEATURE_LABELS.index(label)


def build_true_chromosomal_coefficients():
    beta_C = np.zeros(D_C, dtype=float)
    B = np.zeros((D_C, D_P), dtype=float)

    efflux_machinery = {"acrA", "acrB", "tolC"}
    repressors = {"acrR", "marR"}
    activators = {"marA", "rob", "soxR", "soxS"}
    porins = {"ompC", "ompF"}

    for gene in PREDEFINED_GENES:
        coding_i = feature_index(gene, "coding")
        upstream_i = feature_index(gene, "upstream")

        if gene in efflux_machinery:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in repressors:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene in activators:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in porins:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene == "ampC":
            beta_C[coding_i] = +0.10
            beta_C[upstream_i] = +0.25

        elif gene == "ftsI":
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.10

        if gene in efflux_machinery:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in repressors:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

        elif gene in activators:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in porins:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

    return beta_C, B


BETA_C_TRUE, B_TRUE = build_true_chromosomal_coefficients()


def simulate_targeted_chromosomal_features(rng):
    for _ in range(100):
        C = rng.choice(
            CHROMOSOMAL_STATES,
            size=(N_PATHOGENS, D_C),
            p=CHROMOSOMAL_STATE_PROBS,
        ).astype(float)

        augmented = np.column_stack([
            np.ones(N_PATHOGENS, dtype=float),
            C,
        ])

        if np.linalg.matrix_rank(augmented) == D_C + 1:
            return C

    raise RuntimeError(
        "Could not generate a full-rank pathogen chromosome matrix."
    )


def sample_empirical_plasmids(rng):
    n_empirical = len(empirical_pairs)

    for _ in range(5000):
        selected_positions = rng.choice(
            n_empirical,
            size=N_PLASMIDS,
            replace=False,
        )

        sampled = (
            empirical_pairs.iloc[selected_positions]
            .copy()
            .reset_index(drop=True)
        )

        sampled.insert(
            0,
            "plasmid_id",
            [f"P{j}" for j in range(1, N_PLASMIDS + 1)],
        )

        sampled["I_C32T"] = (
            sampled["sutcliffe_32_nt"].eq("T")
        ).astype(float)

        sampled["I_G162T"] = (
            sampled["sutcliffe_162_nt"].eq("T")
        ).astype(float)

        sampled["I_G175A"] = (
            sampled["sutcliffe_175_nt"].eq("A")
        ).astype(float)

        sampled["q_CN"] = np.log2(
            sampled["CN_TEM1"].astype(float)
            / CN_REFERENCE
        )

        P = np.column_stack([
            np.ones(N_PLASMIDS, dtype=float),
            sampled["I_C32T"].to_numpy(dtype=float),
            sampled["I_G162T"].to_numpy(dtype=float),
            sampled["I_G175A"].to_numpy(dtype=float),
            sampled["q_CN"].to_numpy(dtype=float),
        ])

        P_all = np.vstack([
            np.zeros((1, D_P), dtype=float),
            P,
        ])

        augmented_state_matrix = np.column_stack([
            np.ones(N_PLASMIDS + 1, dtype=float),
            P_all,
        ])

        if np.linalg.matrix_rank(augmented_state_matrix) == D_P + 1:
            return P, sampled

    raise RuntimeError(
        "Could not sample 20 full-rank empirical plasmid profiles."
    )


def simulate_background_relatedness(rng):
    source_frequencies = rng.uniform(
        ALLELE_FREQ_LOW,
        ALLELE_FREQ_HIGH,
        size=N_BACKGROUND_SNPS,
    )

    G = rng.binomial(
        1,
        source_frequencies,
        size=(N_PATHOGENS, N_BACKGROUND_SNPS),
    ).astype(float)

    p = G.mean(axis=0)
    Z = G - p[None, :]

    denominator = float(
        np.sum(p * (1.0 - p))
    )

    if denominator <= 0:
        raise ValueError(
            "Background-SNP relatedness denominator is not positive."
        )

    K = (Z @ Z.T) / denominator
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    if eigenvalues.min() < -1e-8:
        raise ValueError(
            f"Constructed K is unexpectedly non-PSD: "
            f"minimum eigenvalue={eigenvalues.min():.6g}"
        )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    K = (
        eigenvectors * eigenvalues
    ) @ eigenvectors.T

    K = (K + K.T) / 2.0

    return K, G, p


def build_complete_design(C, P):
    P_all = np.vstack([
        np.zeros((1, D_P), dtype=float),
        P,
    ])

    n_states = N_PLASMIDS + 1

    pathogen_index = np.repeat(
        np.arange(N_PATHOGENS, dtype=int),
        n_states,
    )

    plasmid_state_index = np.tile(
        np.arange(n_states, dtype=int),
        N_PATHOGENS,
    )

    C_obs = C[pathogen_index, :]
    P_obs = P_all[plasmid_state_index, :]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    )


def draw_correlated_host_effect(rng, K, sigma_g2):
    eigenvalues, eigenvectors = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    z = rng.normal(
        0.0,
        1.0,
        size=N_PATHOGENS,
    )

    u = eigenvectors @ (
        np.sqrt(
            sigma_g2 * eigenvalues
        ) * z
    )

    return u


def simulate_complete_dataset(seed):
    """
    Generate exactly the same complete biological dataset structure as Simulation 1.
    Missingness is applied only afterwards.
    """
    rng = np.random.default_rng(seed)

    C = simulate_targeted_chromosomal_features(rng)
    P, sampled_plasmids = sample_empirical_plasmids(rng)

    (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    ) = build_complete_design(C, P)

    K, G_background, background_frequencies = (
        simulate_background_relatedness(rng)
    )

    theta_true = np.concatenate([
        np.array([ALPHA_TRUE], dtype=float),
        BETA_C_TRUE,
        BETA_P_TRUE,
        B_TRUE.reshape(-1),
    ])

    structural_mean = X @ theta_true

    u = draw_correlated_host_effect(
        rng,
        K,
        SIGMA_G2_TRUE,
    )

    epsilon = rng.normal(
        0.0,
        SIGMA_E_TRUE,
        size=X.shape[0],
    )

    y = (
        structural_mean
        + u[pathogen_index]
        + epsilon
    )

    return {
        "seed": int(seed),
        "C": C,
        "P": P,
        "P_all": P_all,
        "sampled_plasmids": sampled_plasmids,
        "K": K,
        "G_background": G_background,
        "background_frequencies": background_frequencies,
        "beta_C_true": BETA_C_TRUE.copy(),
        "beta_P_true": BETA_P_TRUE.copy(),
        "B_true": B_TRUE.copy(),
        "theta_true": theta_true,
        "u_true": u,
        "epsilon_true": epsilon,
        "structural_mean": structural_mean,
        "X_complete": X,
        "pathogen_index_complete": pathogen_index,
        "plasmid_state_index_complete": plasmid_state_index,
        "y_complete": y,
    }



def build_joint_support_embedding(C, P):
    """
    Standardize the 60 chromosome features and five plasmid features separately,
    then give the chromosome and plasmid blocks equal aggregate weight.

    The plasmid-presence feature is constant across P+ states, so its standard
    deviation is zero; a scale of 1 is used for such constant features, making
    their pairwise contribution correctly equal to zero.
    """
    C = np.asarray(C, dtype=float)
    P = np.asarray(P, dtype=float)

    c_scale = C.std(
        axis=0,
        ddof=1,
    )

    p_scale = P.std(
        axis=0,
        ddof=1,
    )

    c_scale = np.where(
        c_scale > 1e-12,
        c_scale,
        1.0,
    )

    p_scale = np.where(
        p_scale > 1e-12,
        p_scale,
        1.0,
    )

    C_scaled = (
        C
        / c_scale[None, :]
        / np.sqrt(D_C)
    )

    P_scaled = (
        P
        / p_scale[None, :]
        / np.sqrt(D_P)
    )

    joint_embedding = np.column_stack([
        np.repeat(
            C_scaled,
            N_PLASMIDS,
            axis=0,
        ),
        np.tile(
            P_scaled,
            (N_PATHOGENS, 1),
        ),
    ])

    pathogen_grid_index = np.repeat(
        np.arange(
            N_PATHOGENS,
            dtype=int,
        ),
        N_PLASMIDS,
    )

    plasmid_grid_index = np.tile(
        np.arange(
            N_PLASMIDS,
            dtype=int,
        ),
        N_PATHOGENS,
    )

    return {
        "embedding": joint_embedding,
        "c_scale": c_scale,
        "p_scale": p_scale,
        "pathogen_grid_index": pathogen_grid_index,
        "plasmid_grid_index": plasmid_grid_index,
    }


def calculate_withheld_support(
    C,
    P,
    observed_pplus_grid,
):
    """
    Calculate mean distance from every withheld P+ target to its five nearest
    observed P+ combinations, then classify exactly the lowest and highest
    quartiles as well and poorly supported.
    """
    support_space = build_joint_support_embedding(
        C,
        P,
    )

    embedding = support_space[
        "embedding"
    ]

    observed_flat = (
        observed_pplus_grid.ravel()
    )

    withheld_flat = (
        ~observed_flat
    )

    observed_rows = np.where(
        observed_flat
    )[0]

    withheld_rows = np.where(
        withheld_flat
    )[0]

    if len(observed_rows) < N_SUPPORT_NEIGHBOURS:
        raise RuntimeError(
            "Too few observed P+ combinations for the requested nearest-neighbour support score."
        )

    distances = cdist(
        embedding[
            withheld_rows,
            :
        ],
        embedding[
            observed_rows,
            :
        ],
        metric="euclidean",
    )

    nearest = np.partition(
        distances,
        kth=N_SUPPORT_NEIGHBOURS - 1,
        axis=1,
    )[
        :,
        :N_SUPPORT_NEIGHBOURS,
    ]

    support_distance = nearest.mean(
        axis=1
    )

    n_withheld = len(
        withheld_rows
    )

    n_quartile = int(
        np.floor(
            SUPPORT_QUARTILE_FRACTION
            * n_withheld
        )
    )

    if n_quartile < 1:
        raise RuntimeError(
            "Support quartile contains no targets."
        )

    ordering = np.argsort(
        support_distance,
        kind="mergesort",
    )

    well_local = ordering[
        :n_quartile
    ]

    poor_local = ordering[
        -n_quartile:
    ]

    support_distance_grid = np.full(
        N_PATHOGENS * N_PLASMIDS,
        np.nan,
        dtype=float,
    )

    support_distance_grid[
        withheld_rows
    ] = support_distance

    well_flat = np.zeros(
        N_PATHOGENS * N_PLASMIDS,
        dtype=bool,
    )

    poor_flat = np.zeros(
        N_PATHOGENS * N_PLASMIDS,
        dtype=bool,
    )

    well_flat[
        withheld_rows[
            well_local
        ]
    ] = True

    poor_flat[
        withheld_rows[
            poor_local
        ]
    ] = True

    return {
        "support_distance_grid":
            support_distance_grid.reshape(
                N_PATHOGENS,
                N_PLASMIDS,
            ),

        "well_supported_grid":
            well_flat.reshape(
                N_PATHOGENS,
                N_PLASMIDS,
            ),

        "poorly_supported_grid":
            poor_flat.reshape(
                N_PATHOGENS,
                N_PLASMIDS,
            ),

        "n_withheld_targets":
            int(
                n_withheld
            ),

        "n_targets_per_primary_quartile":
            int(
                n_quartile
            ),

        "well_support_distance_mean":
            float(
                support_distance[
                    well_local
                ].mean()
            ),

        "poor_support_distance_mean":
            float(
                support_distance[
                    poor_local
                ].mean()
            ),

        "well_support_distance_max":
            float(
                support_distance[
                    well_local
                ].max()
            ),

        "poor_support_distance_min":
            float(
                support_distance[
                    poor_local
                ].min()
            ),
    }


def apply_weak_support_observation_pattern(
    dataset,
    mask_seed,
):
    """
    Withhold 1,200 P+ combinations:
      - 600 nearest to 10 randomly selected joint chromosome-plasmid centres;
      - 600 additional combinations selected at random from the remainder.

    Redraw the observation pattern until:
      - all P0 states remain observed;
      - every pathogen retains at least one observed P+ state;
      - every plasmid remains observed;
      - the 366-column fixed-effect design is full rank.
    """
    X_complete = dataset[
        "X_complete"
    ]

    p_fixed = X_complete.shape[1]

    support_space = build_joint_support_embedding(
        dataset["C"],
        dataset["P"],
    )

    embedding = support_space[
        "embedding"
    ]

    n_pplus = (
        N_PATHOGENS
        * N_PLASMIDS
    )

    rng = np.random.default_rng(
        mask_seed
    )

    for attempt in range(
        1,
        501,
    ):
        centre_rows = rng.choice(
            n_pplus,
            size=N_CLUSTER_CENTRES,
            replace=False,
        )

        centre_distances = cdist(
            embedding,
            embedding[
                centre_rows,
                :
            ],
            metric="euclidean",
        ).min(
            axis=1
        )

        # Deterministic nearest-neighbour clustered component for these centres.
        clustered_rows = np.argsort(
            centre_distances,
            kind="mergesort",
        )[
            :N_CLUSTERED_WITHHELD
        ]

        remaining_rows = np.setdiff1d(
            np.arange(
                n_pplus,
                dtype=int,
            ),
            clustered_rows,
            assume_unique=False,
        )

        random_rows = rng.choice(
            remaining_rows,
            size=N_RANDOM_WITHHELD,
            replace=False,
        )

        missing_flat = np.zeros(
            n_pplus,
            dtype=bool,
        )

        missing_flat[
            clustered_rows
        ] = True

        missing_flat[
            random_rows
        ] = True

        observed_pplus_grid = (
            ~missing_flat
        ).reshape(
            N_PATHOGENS,
            N_PLASMIDS,
        )

        # Every pathogen and every plasmid must remain represented.
        if np.any(
            observed_pplus_grid.sum(
                axis=1
            ) == 0
        ):
            continue

        if np.any(
            observed_pplus_grid.sum(
                axis=0
            ) == 0
        ):
            continue

        grid21 = np.column_stack([
            np.ones(
                N_PATHOGENS,
                dtype=bool,
            ),
            observed_pplus_grid,
        ])

        observed_mask = (
            grid21.ravel()
        )

        X_observed = X_complete[
            observed_mask,
            :,
        ]

        if np.linalg.matrix_rank(
            X_observed
        ) != p_fixed:
            continue

        support = calculate_withheld_support(
            dataset["C"],
            dataset["P"],
            observed_pplus_grid,
        )

        centre_pathogens = (
            centre_rows
            // N_PLASMIDS
        )

        centre_plasmids = (
            centre_rows
            % N_PLASMIDS
        )

        return {
            "observed_mask":
                observed_mask,

            "observed_pplus_grid":
                observed_pplus_grid,

            "missing_pplus_grid":
                ~observed_pplus_grid,

            "clustered_withheld_grid":
                np.isin(
                    np.arange(
                        n_pplus
                    ),
                    clustered_rows,
                ).reshape(
                    N_PATHOGENS,
                    N_PLASMIDS,
                ),

            "random_withheld_grid":
                np.isin(
                    np.arange(
                        n_pplus
                    ),
                    random_rows,
                ).reshape(
                    N_PATHOGENS,
                    N_PLASMIDS,
                ),

            "cluster_centre_rows":
                centre_rows.copy(),

            "cluster_centre_pathogen_ids": [
                f"C{i + 1}"
                for i in centre_pathogens
            ],

            "cluster_centre_plasmid_ids": [
                f"P{j + 1}"
                for j in centre_plasmids
            ],

            "mask_attempt":
                int(
                    attempt
                ),

            "n_missing_pplus":
                int(
                    missing_flat.sum()
                ),

            "n_observed_pplus":
                int(
                    observed_pplus_grid.sum()
                ),

            **support,
        }

    raise RuntimeError(
        "Could not generate a valid Simulation 6 observation pattern "
        "after 500 attempts."
    )


def prepare_support_dataset(seed):
    dataset = simulate_complete_dataset(
        seed
    )

    observation_pattern = (
        apply_weak_support_observation_pattern(
            dataset,
            mask_seed=seed + 60_000_000,
        )
    )

    observed_mask = observation_pattern[
        "observed_mask"
    ]

    dataset.update(
        observation_pattern
    )

    dataset["X"] = dataset[
        "X_complete"
    ][
        observed_mask,
        :
    ]

    dataset["y"] = dataset[
        "y_complete"
    ][
        observed_mask
    ]

    dataset["pathogen_index"] = dataset[
        "pathogen_index_complete"
    ][
        observed_mask
    ]

    dataset["plasmid_state_index"] = dataset[
        "plasmid_state_index_complete"
    ][
        observed_mask
    ]

    return dataset


print("Cell 3: PASS")


In [ ]:
#@title Cell 4 - Define efficient REML and GLS fitting

def prepare_reml_static(X, pathogen_index, K):
    X = np.asarray(X, dtype=float)
    K = np.asarray(K, dtype=float)

    m, p_fixed = X.shape

    design_rank = np.linalg.matrix_rank(X)

    if design_rank != p_fixed:
        raise ValueError(
            f"Fixed-effect design matrix is rank deficient: "
            f"rank={design_rank}, columns={p_fixed}."
        )

    eigenvalues, Q = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    keep = eigenvalues > 1e-10
    eigenvalues = eigenvalues[keep]
    Q = Q[:, keep]

    B_lowrank = (
        Q[pathogen_index, :]
        * np.sqrt(eigenvalues)[None, :]
    )

    static = {
        "m": int(m),
        "p_fixed": int(p_fixed),
        "rank_K": int(len(eigenvalues)),
        "B_lowrank": B_lowrank,
        "BtB": B_lowrank.T @ B_lowrank,
        "BtX": B_lowrank.T @ X,
        "XTX": X.T @ X,
        "X": X,
    }

    return static


def add_y_to_reml_static(static, y):
    working = {
        key: value
        for key, value in static.items()
        if key not in {"B_lowrank", "X"}
    }

    B_lowrank = static["B_lowrank"]
    X = static["X"]

    working["Bty"] = B_lowrank.T @ y
    working["Xty"] = X.T @ y
    working["yty"] = float(y @ y)

    return working


def evaluate_profile_reml(log_delta, working, return_fit=False):
    delta = float(np.exp(log_delta))

    m = working["m"]
    p_fixed = working["p_fixed"]
    rank_K = working["rank_K"]

    M = (
        np.eye(rank_K)
        + working["BtB"] / delta
    )

    try:
        chol_M = cho_factor(
            M,
            lower=True,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    M_inv_BtX = cho_solve(
        chol_M,
        working["BtX"],
        check_finite=False,
    )

    M_inv_Bty = cho_solve(
        chol_M,
        working["Bty"],
        check_finite=False,
    )

    XtAinvX = (
        working["XTX"] / delta
        - (
            working["BtX"].T
            @ M_inv_BtX
        ) / (delta ** 2)
    )

    XtAinvy = (
        working["Xty"] / delta
        - (
            working["BtX"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    yAinvy = (
        working["yty"] / delta
        - float(
            working["Bty"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    XtAinvX = (
        XtAinvX + XtAinvX.T
    ) / 2.0

    sign_X, logdet_X = np.linalg.slogdet(
        XtAinvX
    )

    if sign_X <= 0:
        return np.inf if not return_fit else None

    try:
        chol_X = cho_factor(
            XtAinvX,
            lower=True,
            check_finite=False,
        )

        beta_hat = cho_solve(
            chol_X,
            XtAinvy,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    q = float(
        yAinvy
        - beta_hat @ XtAinvy
    )

    df_reml = m - p_fixed

    if q <= 0 or df_reml <= 0:
        return np.inf if not return_fit else None

    logdet_M = 2.0 * np.sum(
        np.log(np.diag(chol_M[0]))
    )

    logdet_A = (
        m * np.log(delta)
        + logdet_M
    )

    objective = (
        logdet_A
        + logdet_X
        + df_reml * np.log(q / df_reml)
    )

    if return_fit:
        return {
            "objective": float(objective),
            "beta_hat": beta_hat,
            "q": q,
            "delta": delta,
            "df_reml": int(df_reml),
        }

    return float(objective)


def fit_section2_reml_gls(X, y, pathogen_index, K, static=None):
    if static is None:
        static = prepare_reml_static(
            X,
            pathogen_index,
            K,
        )

    working = add_y_to_reml_static(
        static,
        y,
    )

    optimization = minimize_scalar(
        lambda log_delta: evaluate_profile_reml(
            log_delta,
            working,
            return_fit=False,
        ),
        bounds=(-8.0, 8.0),
        method="bounded",
        options={
            "xatol": 1e-4,
            "maxiter": 100,
        },
    )

    if not optimization.success:
        raise RuntimeError(
            "REML optimization failed: "
            + str(optimization.message)
        )

    fit = evaluate_profile_reml(
        optimization.x,
        working,
        return_fit=True,
    )

    if fit is None:
        raise RuntimeError(
            "Final REML/GLS evaluation failed."
        )

    sigma_g2_hat = (
        fit["q"]
        / fit["df_reml"]
    )

    sigma_e2_hat = (
        fit["delta"]
        * sigma_g2_hat
    )

    return {
        "beta_hat": fit["beta_hat"],
        "sigma_g2_hat": float(sigma_g2_hat),
        "sigma_e2_hat": float(sigma_e2_hat),
        "delta_hat": float(fit["delta"]),
        "reml_objective": float(fit["objective"]),
        "optimization_nfev": int(optimization.nfev),
        "static": static,
    }


def unpack_beta(beta_hat):
    start_C = 1
    stop_C = start_C + D_C

    start_P = stop_C
    stop_P = start_P + D_P

    start_B = stop_P

    alpha_hat = float(beta_hat[0])
    beta_C_hat = beta_hat[start_C:stop_C]
    beta_P_hat = beta_hat[start_P:stop_P]
    B_hat = beta_hat[start_B:].reshape(
        D_C,
        D_P,
    )

    return (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    )


print("Cell 4: PASS")


In [ ]:
#@title Cell 5 - Generate and QC one Simulation 6 dataset

EXAMPLE_SEED = MASTER_SEED

example = prepare_support_dataset(
    EXAMPLE_SEED
)

X = example["X"]
y = example["y"]
K = example["K"]

mask_table = pd.DataFrame(
    np.column_stack([
        np.ones(
            N_PATHOGENS,
            dtype=int,
        ),
        example[
            "observed_pplus_grid"
        ].astype(int),
    ]),
    index=[
        f"C{i}"
        for i in range(
            1,
            N_PATHOGENS + 1,
        )
    ],
    columns=[
        "P0",
        *[
            f"P{j}"
            for j in range(
                1,
                N_PLASMIDS + 1,
            )
        ],
    ],
)

MASK_PATH = (
    OUTPUT_DIR
    / "06_example_observation_mask_200x21.csv"
)

mask_table.to_csv(
    MASK_PATH,
    index=True,
)

support_rows = []

for i in range(
    N_PATHOGENS
):
    for j in range(
        N_PLASMIDS
    ):
        if not example[
            "missing_pplus_grid"
        ][
            i,
            j,
        ]:
            continue

        if example[
            "well_supported_grid"
        ][
            i,
            j,
        ]:
            group = "well_supported"
        elif example[
            "poorly_supported_grid"
        ][
            i,
            j,
        ]:
            group = "poorly_supported"
        else:
            group = "middle_50_percent"

        construction = (
            "clustered"
            if example[
                "clustered_withheld_grid"
            ][
                i,
                j,
            ]
            else "random"
        )

        support_rows.append({
            "pathogen_id":
                f"C{i + 1}",

            "plasmid_id":
                f"P{j + 1}",

            "construction":
                construction,

            "support_group":
                group,

            "mean_5NN_distance":
                example[
                    "support_distance_grid"
                ][
                    i,
                    j,
                ],
        })

SUPPORT_TABLE_PATH = (
    OUTPUT_DIR
    / "06_example_withheld_target_support.csv"
)

pd.DataFrame(
    support_rows
).to_csv(
    SUPPORT_TABLE_PATH,
    index=False,
)

print("=" * 90)
print("CELL 5 — SIMULATION 6 OBSERVATION-PATTERN AND SUPPORT QC")
print("=" * 90)

print(
    f"Complete observations before masking: "
    f"{N_PATHOGENS * (N_PLASMIDS + 1):,}"
)

print(
    f"Observed rows:                        "
    f"{len(y):,}"
)

print(
    f"Observed P0 states:                   "
    f"{N_PATHOGENS}/{N_PATHOGENS}"
)

print(
    f"Observed P+ combinations:             "
    f"{example['n_observed_pplus']:,}/"
    f"{N_PATHOGENS * N_PLASMIDS:,}"
)

print(
    f"Withheld P+ combinations:             "
    f"{example['n_missing_pplus']:,}"
)

print(
    f"Clustered withheld:                   "
    f"{int(example['clustered_withheld_grid'].sum()):,}"
)

print(
    f"Random withheld:                      "
    f"{int(example['random_withheld_grid'].sum()):,}"
)

print(
    f"Mask generation attempt:              "
    f"{example['mask_attempt']}"
)

print(
    f"Fixed-effect columns:                 "
    f"{X.shape[1]}"
)

print(
    f"Observed design rank:                 "
    f"{np.linalg.matrix_rank(X)}"
)

print(
    f"K dimensions:                        "
    f"{K.shape}"
)

print(
    f"P+ observations per pathogen:        "
    f"min={example['observed_pplus_grid'].sum(axis=1).min()}, "
    f"mean={example['observed_pplus_grid'].sum(axis=1).mean():.2f}, "
    f"max={example['observed_pplus_grid'].sum(axis=1).max()}"
)

print(
    f"Pathogen observations per plasmid:    "
    f"min={example['observed_pplus_grid'].sum(axis=0).min()}, "
    f"mean={example['observed_pplus_grid'].sum(axis=0).mean():.2f}, "
    f"max={example['observed_pplus_grid'].sum(axis=0).max()}"
)

print("\nSupport diagnostics:")
print(
    f"Withheld targets:                     "
    f"{example['n_withheld_targets']}"
)

print(
    f"Targets per primary quartile:         "
    f"{example['n_targets_per_primary_quartile']}"
)

print(
    f"Mean 5NN distance, well supported:    "
    f"{example['well_support_distance_mean']:.6f}"
)

print(
    f"Mean 5NN distance, poorly supported:  "
    f"{example['poor_support_distance_mean']:.6f}"
)

print(
    f"Well-support maximum distance:        "
    f"{example['well_support_distance_max']:.6f}"
)

print(
    f"Poor-support minimum distance:        "
    f"{example['poor_support_distance_min']:.6f}"
)

print("\nCluster centres:")
for c_id, p_id in zip(
    example["cluster_centre_pathogen_ids"],
    example["cluster_centre_plasmid_ids"],
):
    print(
        f"  {c_id} × {p_id}"
    )

if (
    example["n_missing_pplus"]
    != N_CLUSTERED_WITHHELD
    + N_RANDOM_WITHHELD
):
    raise RuntimeError(
        "Unexpected number of withheld P+ combinations."
    )

if (
    np.linalg.matrix_rank(X)
    != X.shape[1]
):
    raise RuntimeError(
        "Observed fixed-effect design is not full rank."
    )

if not (
    example["poor_support_distance_mean"]
    > example["well_support_distance_mean"]
):
    raise RuntimeError(
        "Poor-support targets do not have larger support distances."
    )

print("\nSaved:")
print(MASK_PATH)
print(SUPPORT_TABLE_PATH)

print("\nCell 5: PASS")


In [ ]:
#@title Cell 6 - Fit the model using the observed Simulation 6 dataset

start_time = time.time()

example_static = prepare_reml_static(
    example["X"],
    example["pathogen_index"],
    example["K"],
)

example_fit = fit_section2_reml_gls(
    example["X"],
    example["y"],
    example["pathogen_index"],
    example["K"],
    static=example_static,
)

elapsed = (
    time.time()
    - start_time
)

(
    alpha_hat,
    beta_C_hat,
    beta_P_hat,
    B_hat,
) = unpack_beta(
    example_fit[
        "beta_hat"
    ]
)

print("=" * 90)
print("CELL 6 — SIMULATION 6 REML/GLS FIT")
print("=" * 90)

print(
    f"Observed rows fitted:          "
    f"{len(example['y']):,}"
)

print(
    f"True alpha:                    "
    f"{ALPHA_TRUE:.6f}"
)

print(
    f"Estimated alpha:               "
    f"{alpha_hat:.6f}"
)

print(
    f"True sigma_g^2:                "
    f"{SIGMA_G2_TRUE:.6f}"
)

print(
    f"Estimated sigma_g^2:           "
    f"{example_fit['sigma_g2_hat']:.6f}"
)

print(
    f"True sigma_e^2:                "
    f"{SIGMA_E2_TRUE:.6f}"
)

print(
    f"Estimated sigma_e^2:           "
    f"{example_fit['sigma_e2_hat']:.6f}"
)

print(
    f"Estimated variance ratio:      "
    f"{example_fit['delta_hat']:.6f}"
)

print(
    f"Fit time:                      "
    f"{elapsed:.2f} seconds"
)

plasmid_recovery = pd.DataFrame({
    "feature":
        PLASMID_FEATURE_LABELS,

    "true_beta_P":
        BETA_P_TRUE,

    "estimated_beta_P":
        beta_P_hat,
})

print("\nPlasmid main-effect recovery:")
display(
    plasmid_recovery
)

print("\nCell 6: PASS")


In [ ]:
#@title Cell 7 - Evaluate recovery by local support

def true_and_estimated_effects(
    dataset,
    fit,
):
    C = dataset["C"]
    P = dataset["P"]

    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        fit["beta_hat"]
    )

    y0_true = (
        ALPHA_TRUE
        + C
        @ dataset[
            "beta_C_true"
        ]
    )

    delta_true = (
        P
        @ dataset[
            "beta_P_true"
        ]
    )[
        None,
        :
    ] + (
        C
        @ dataset[
            "B_true"
        ]
        @ P.T
    )

    y0_hat = (
        alpha_hat
        + C
        @ beta_C_hat
    )

    delta_hat = (
        P
        @ beta_P_hat
    )[
        None,
        :
    ] + (
        C
        @ B_hat
        @ P.T
    )

    return {
        "y0_true":
            y0_true,

        "y0_hat":
            y0_hat,

        "delta_true":
            delta_true,

        "delta_hat":
            delta_hat,
    }


def basic_metrics(
    true_values,
    estimated_values,
):
    true_values = np.asarray(
        true_values,
        dtype=float,
    ).ravel()

    estimated_values = np.asarray(
        estimated_values,
        dtype=float,
    ).ravel()

    error = (
        estimated_values
        - true_values
    )

    nonzero = (
        np.abs(
            true_values
        )
        > 1e-12
    )

    if nonzero.any():
        sign_accuracy = np.mean(
            np.sign(
                estimated_values[
                    nonzero
                ]
            )
            == np.sign(
                true_values[
                    nonzero
                ]
            )
        )
    else:
        sign_accuracy = np.nan

    return {
        "bias":
            float(
                np.mean(
                    error
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "sign_accuracy":
            float(
                sign_accuracy
            ),
    }


def build_same_plasmid_dd_targets(
    support_target_grid,
    observed_pplus_grid,
):
    """
    For each withheld support target (i,j), compare Delta_ij with Delta_kj
    for every pathogen k where the same plasmid j is actually observed.
    """
    target_i = []
    comparison_k = []
    plasmid_j = []

    for j in range(
        N_PLASMIDS
    ):
        targets = np.where(
            support_target_grid[
                :,
                j,
            ]
        )[0]

        observed_hosts = np.where(
            observed_pplus_grid[
                :,
                j,
            ]
        )[0]

        for i in targets:
            target_i.extend(
                [int(i)]
                * len(
                    observed_hosts
                )
            )

            comparison_k.extend(
                observed_hosts.astype(
                    int
                ).tolist()
            )

            plasmid_j.extend(
                [int(j)]
                * len(
                    observed_hosts
                )
            )

    return (
        np.asarray(
            target_i,
            dtype=int,
        ),
        np.asarray(
            comparison_k,
            dtype=int,
        ),
        np.asarray(
            plasmid_j,
            dtype=int,
        ),
    )


def evaluate_support_recovery(
    dataset,
    fit,
):
    effects = true_and_estimated_effects(
        dataset,
        fit,
    )

    delta_true = effects[
        "delta_true"
    ]

    delta_hat = effects[
        "delta_hat"
    ]

    well_grid = dataset[
        "well_supported_grid"
    ]

    poor_grid = dataset[
        "poorly_supported_grid"
    ]

    well_delta_metrics = basic_metrics(
        delta_true[
            well_grid
        ],
        delta_hat[
            well_grid
        ],
    )

    poor_delta_metrics = basic_metrics(
        delta_true[
            poor_grid
        ],
        delta_hat[
            poor_grid
        ],
    )

    (
        well_i,
        well_k,
        well_j,
    ) = build_same_plasmid_dd_targets(
        well_grid,
        dataset[
            "observed_pplus_grid"
        ],
    )

    (
        poor_i,
        poor_k,
        poor_j,
    ) = build_same_plasmid_dd_targets(
        poor_grid,
        dataset[
            "observed_pplus_grid"
        ],
    )

    well_dd_true = (
        delta_true[
            well_i,
            well_j,
        ]
        - delta_true[
            well_k,
            well_j,
        ]
    )

    well_dd_hat = (
        delta_hat[
            well_i,
            well_j,
        ]
        - delta_hat[
            well_k,
            well_j,
        ]
    )

    poor_dd_true = (
        delta_true[
            poor_i,
            poor_j,
        ]
        - delta_true[
            poor_k,
            poor_j,
        ]
    )

    poor_dd_hat = (
        delta_hat[
            poor_i,
            poor_j,
        ]
        - delta_hat[
            poor_k,
            poor_j,
        ]
    )

    well_dd_metrics = basic_metrics(
        well_dd_true,
        well_dd_hat,
    )

    poor_dd_metrics = basic_metrics(
        poor_dd_true,
        poor_dd_hat,
    )

    delta_rmse_ratio = (
        poor_delta_metrics[
            "rmse"
        ]
        / well_delta_metrics[
            "rmse"
        ]
    )

    dd_rmse_ratio = (
        poor_dd_metrics[
            "rmse"
        ]
        / well_dd_metrics[
            "rmse"
        ]
    )

    row = {
        "well_delta_bias":
            well_delta_metrics[
                "bias"
            ],

        "well_delta_rmse":
            well_delta_metrics[
                "rmse"
            ],

        "well_delta_sign_accuracy":
            well_delta_metrics[
                "sign_accuracy"
            ],

        "poor_delta_bias":
            poor_delta_metrics[
                "bias"
            ],

        "poor_delta_rmse":
            poor_delta_metrics[
                "rmse"
            ],

        "poor_delta_sign_accuracy":
            poor_delta_metrics[
                "sign_accuracy"
            ],

        "well_delta_delta_bias":
            well_dd_metrics[
                "bias"
            ],

        "well_delta_delta_rmse":
            well_dd_metrics[
                "rmse"
            ],

        "well_delta_delta_sign_accuracy":
            well_dd_metrics[
                "sign_accuracy"
            ],

        "poor_delta_delta_bias":
            poor_dd_metrics[
                "bias"
            ],

        "poor_delta_delta_rmse":
            poor_dd_metrics[
                "rmse"
            ],

        "poor_delta_delta_sign_accuracy":
            poor_dd_metrics[
                "sign_accuracy"
            ],

        "delta_rmse_ratio_poor_over_well":
            float(
                delta_rmse_ratio
            ),

        "delta_delta_rmse_ratio_poor_over_well":
            float(
                dd_rmse_ratio
            ),

        "n_well_delta_targets":
            int(
                well_grid.sum()
            ),

        "n_poor_delta_targets":
            int(
                poor_grid.sum()
            ),

        "n_well_delta_delta_targets":
            int(
                len(
                    well_dd_true
                )
            ),

        "n_poor_delta_delta_targets":
            int(
                len(
                    poor_dd_true
                )
            ),

        "well_support_distance_mean":
            dataset[
                "well_support_distance_mean"
            ],

        "poor_support_distance_mean":
            dataset[
                "poor_support_distance_mean"
            ],

        "sigma_g2_hat":
            fit[
                "sigma_g2_hat"
            ],

        "sigma_e2_hat":
            fit[
                "sigma_e2_hat"
            ],
    }

    return {
        "metrics":
            row,

        "effects":
            effects,

        "well_dd_true":
            well_dd_true,

        "well_dd_hat":
            well_dd_hat,

        "poor_dd_true":
            poor_dd_true,

        "poor_dd_hat":
            poor_dd_hat,

        "well_dd_indices":
            (
                well_i,
                well_k,
                well_j,
            ),

        "poor_dd_indices":
            (
                poor_i,
                poor_k,
                poor_j,
            ),
    }


example_evaluation = evaluate_support_recovery(
    example,
    example_fit,
)

example_metrics = example_evaluation[
    "metrics"
]

print("=" * 90)
print("CELL 7 — RECOVERY BY LOCAL SUPPORT")
print("=" * 90)

display(
    pd.DataFrame([
        {
            "support_group":
                "well_supported",

            "Delta_bias":
                example_metrics[
                    "well_delta_bias"
                ],

            "Delta_RMSE":
                example_metrics[
                    "well_delta_rmse"
                ],

            "Delta_sign_accuracy":
                example_metrics[
                    "well_delta_sign_accuracy"
                ],

            "DeltaDelta_bias":
                example_metrics[
                    "well_delta_delta_bias"
                ],

            "DeltaDelta_RMSE":
                example_metrics[
                    "well_delta_delta_rmse"
                ],

            "DeltaDelta_sign_accuracy":
                example_metrics[
                    "well_delta_delta_sign_accuracy"
                ],
        },
        {
            "support_group":
                "poorly_supported",

            "Delta_bias":
                example_metrics[
                    "poor_delta_bias"
                ],

            "Delta_RMSE":
                example_metrics[
                    "poor_delta_rmse"
                ],

            "Delta_sign_accuracy":
                example_metrics[
                    "poor_delta_sign_accuracy"
                ],

            "DeltaDelta_bias":
                example_metrics[
                    "poor_delta_delta_bias"
                ],

            "DeltaDelta_RMSE":
                example_metrics[
                    "poor_delta_delta_rmse"
                ],

            "DeltaDelta_sign_accuracy":
                example_metrics[
                    "poor_delta_delta_sign_accuracy"
                ],
        },
    ])
)

print("\nPrimary Simulation 6 comparison:")

print(
    f"Delta RMSE ratio, poor/well:      "
    f"{example_metrics['delta_rmse_ratio_poor_over_well']:.3f}"
)

print(
    f"DeltaDelta RMSE ratio, poor/well: "
    f"{example_metrics['delta_delta_rmse_ratio_poor_over_well']:.3f}"
)

print(
    f"Starting meaningful-loss threshold: "
    f">{GO_RMSE_RATIO:.2f}"
)

print(
    f"\nDelta targets: "
    f"well={example_metrics['n_well_delta_targets']}, "
    f"poor={example_metrics['n_poor_delta_targets']}"
)

print(
    f"DeltaDelta targets: "
    f"well={example_metrics['n_well_delta_delta_targets']:,}, "
    f"poor={example_metrics['n_poor_delta_delta_targets']:,}"
)

print("\nCell 7: PASS")


In [ ]:
#@title Cell 8 - Run 100 independent Simulation 6 replicates

def run_one_simulation6_replicate(
    replicate_index,
):
    seed = (
        MASTER_SEED
        + 6000
        + int(
            replicate_index
        )
    )

    dataset = prepare_support_dataset(
        seed
    )

    static = prepare_reml_static(
        dataset["X"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    fit = fit_section2_reml_gls(
        dataset["X"],
        dataset["y"],
        dataset["pathogen_index"],
        dataset["K"],
        static=static,
    )

    evaluation = evaluate_support_recovery(
        dataset,
        fit,
    )

    metrics = evaluation[
        "metrics"
    ].copy()

    metrics[
        "replicate"
    ] = int(
        replicate_index
        + 1
    )

    metrics[
        "seed"
    ] = int(
        seed
    )

    metrics[
        "mask_attempt"
    ] = int(
        dataset[
            "mask_attempt"
        ]
    )

    metrics[
        "min_Pplus_per_pathogen"
    ] = int(
        dataset[
            "observed_pplus_grid"
        ].sum(
            axis=1
        ).min()
    )

    metrics[
        "min_pathogens_per_plasmid"
    ] = int(
        dataset[
            "observed_pplus_grid"
        ].sum(
            axis=0
        ).min()
    )

    return metrics


replicate_rows = []

start_time = time.time()

for r in range(
    N_SIM_REPLICATES
):
    print(
        f"Starting replicate "
        f"{r + 1:3d}/{N_SIM_REPLICATES}..."
    )

    replicate_start = (
        time.time()
    )

    replicate_rows.append(
        run_one_simulation6_replicate(
            r
        )
    )

    replicate_elapsed = (
        time.time()
        - replicate_start
    )

    print(
        f"Finished replicate "
        f"{r + 1:3d}/{N_SIM_REPLICATES} "
        f"in {replicate_elapsed:.1f} seconds"
    )

    if (
        (r + 1) % 10 == 0
        or r == 0
        or r + 1 == N_SIM_REPLICATES
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {r + 1:3d}/{N_SIM_REPLICATES} replicates "
            f"({elapsed:.1f} seconds elapsed)"
        )

replicate_results = pd.DataFrame(
    replicate_rows
)

REPLICATE_RESULTS_PATH = (
    OUTPUT_DIR
    / "06_scenario6_support_replicate_metrics.csv"
)

replicate_results.to_csv(
    REPLICATE_RESULTS_PATH,
    index=False,
)

print("\nSaved:")
print(
    REPLICATE_RESULTS_PATH
)

print("\nCell 8: PASS")


In [ ]:
#@title Cell 9 - Summarize the 100-replicate Simulation 6 benchmark

PRIMARY_METRICS = [
    "well_delta_bias",
    "well_delta_rmse",
    "well_delta_sign_accuracy",
    "poor_delta_bias",
    "poor_delta_rmse",
    "poor_delta_sign_accuracy",
    "well_delta_delta_bias",
    "well_delta_delta_rmse",
    "well_delta_delta_sign_accuracy",
    "poor_delta_delta_bias",
    "poor_delta_delta_rmse",
    "poor_delta_delta_sign_accuracy",
    "delta_rmse_ratio_poor_over_well",
    "delta_delta_rmse_ratio_poor_over_well",
]

SUPPORTING_METRICS = [
    "well_support_distance_mean",
    "poor_support_distance_mean",
    "sigma_g2_hat",
    "sigma_e2_hat",
    "mask_attempt",
    "min_Pplus_per_pathogen",
    "min_pathogens_per_plasmid",
]

summary_rows = []

for metric in (
    PRIMARY_METRICS
    + SUPPORTING_METRICS
):
    values = replicate_results[
        metric
    ].to_numpy(
        dtype=float
    )

    summary_rows.append({
        "metric":
            metric,

        "mean":
            float(
                np.nanmean(
                    values
                )
            ),

        "sd":
            float(
                np.nanstd(
                    values,
                    ddof=1,
                )
            ),

        "median":
            float(
                np.nanmedian(
                    values
                )
            ),

        "q025":
            float(
                np.nanquantile(
                    values,
                    0.025,
                )
            ),

        "q975":
            float(
                np.nanquantile(
                    values,
                    0.975,
                )
            ),
    })

summary = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "06_scenario6_weak_support_summary.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

mean_well_delta_rmse = float(
    replicate_results[
        "well_delta_rmse"
    ].mean()
)

mean_poor_delta_rmse = float(
    replicate_results[
        "poor_delta_rmse"
    ].mean()
)

mean_well_dd_rmse = float(
    replicate_results[
        "well_delta_delta_rmse"
    ].mean()
)

mean_poor_dd_rmse = float(
    replicate_results[
        "poor_delta_delta_rmse"
    ].mean()
)

ratio_of_mean_delta_rmse = (
    mean_poor_delta_rmse
    / mean_well_delta_rmse
)

ratio_of_mean_dd_rmse = (
    mean_poor_dd_rmse
    / mean_well_dd_rmse
)

fraction_delta_poor_worse = float(
    np.mean(
        replicate_results[
            "poor_delta_rmse"
        ]
        > replicate_results[
            "well_delta_rmse"
        ]
    )
)

fraction_dd_poor_worse = float(
    np.mean(
        replicate_results[
            "poor_delta_delta_rmse"
        ]
        > replicate_results[
            "well_delta_delta_rmse"
        ]
    )
)

fraction_delta_ratio_over_110 = float(
    np.mean(
        replicate_results[
            "delta_rmse_ratio_poor_over_well"
        ]
        > GO_RMSE_RATIO
    )
)

fraction_dd_ratio_over_110 = float(
    np.mean(
        replicate_results[
            "delta_delta_rmse_ratio_poor_over_well"
        ]
        > GO_RMSE_RATIO
    )
)

print("=" * 90)
print("SCENARIO 6 — 100-REPLICATE SUMMARY")
print("=" * 90)

print("\nPrimary comparison:")
display(
    summary[
        summary[
            "metric"
        ].isin(
            PRIMARY_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSupport and variance diagnostics:")
display(
    summary[
        summary[
            "metric"
        ].isin(
            SUPPORTING_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nDirect loss from poor local support:")

print(
    f"Ratio of mean Delta RMSE, poor/well:      "
    f"{ratio_of_mean_delta_rmse:.3f}"
)

print(
    f"Ratio of mean DeltaDelta RMSE, poor/well: "
    f"{ratio_of_mean_dd_rmse:.3f}"
)

print(
    f"Replicates with poorer-support Delta RMSE higher:      "
    f"{fraction_delta_poor_worse:.1%}"
)

print(
    f"Replicates with poorer-support DeltaDelta RMSE higher: "
    f"{fraction_dd_poor_worse:.1%}"
)

print(
    f"Replicates with Delta RMSE ratio > {GO_RMSE_RATIO:.2f}:      "
    f"{fraction_delta_ratio_over_110:.1%}"
)

print(
    f"Replicates with DeltaDelta RMSE ratio > {GO_RMSE_RATIO:.2f}: "
    f"{fraction_dd_ratio_over_110:.1%}"
)

print("\nSaved:")
print(
    SUMMARY_PATH
)

print("\nCell 9: PASS")


In [ ]:
#@title Cell 10 - Parametric bootstrap by local-support group

def simulate_parametric_bootstrap_y(
    rng,
    X,
    pathogen_index,
    K,
    beta_hat,
    sigma_g2_hat,
    sigma_e2_hat,
):
    u_star = draw_correlated_host_effect(
        rng,
        K,
        sigma_g2_hat,
    )

    epsilon_star = rng.normal(
        0.0,
        np.sqrt(
            sigma_e2_hat
        ),
        size=X.shape[0],
    )

    return (
        X
        @ beta_hat
        + u_star[
            pathogen_index
        ]
        + epsilon_star
    )


def effects_from_beta(
    C,
    P,
    beta_hat,
):
    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        beta_hat
    )

    y0_hat = (
        alpha_hat
        + C
        @ beta_C_hat
    )

    delta_hat = (
        P
        @ beta_P_hat
    )[
        None,
        :
    ] + (
        C
        @ B_hat
        @ P.T
    )

    return (
        y0_hat,
        delta_hat,
    )


well_delta_mask = example[
    "well_supported_grid"
]

poor_delta_mask = example[
    "poorly_supported_grid"
]

delta_true = example_evaluation[
    "effects"
][
    "delta_true"
]

well_delta_true = delta_true[
    well_delta_mask
]

poor_delta_true = delta_true[
    poor_delta_mask
]

(
    well_i,
    well_k,
    well_j,
) = example_evaluation[
    "well_dd_indices"
]

(
    poor_i,
    poor_k,
    poor_j,
) = example_evaluation[
    "poor_dd_indices"
]

well_dd_true = example_evaluation[
    "well_dd_true"
]

poor_dd_true = example_evaluation[
    "poor_dd_true"
]

bootstrap_well_delta = np.empty(
    (
        N_BOOTSTRAP,
        len(
            well_delta_true
        ),
    ),
    dtype=np.float32,
)

bootstrap_poor_delta = np.empty(
    (
        N_BOOTSTRAP,
        len(
            poor_delta_true
        ),
    ),
    dtype=np.float32,
)

bootstrap_well_dd = np.empty(
    (
        N_BOOTSTRAP,
        len(
            well_dd_true
        ),
    ),
    dtype=np.float32,
)

bootstrap_poor_dd = np.empty(
    (
        N_BOOTSTRAP,
        len(
            poor_dd_true
        ),
    ),
    dtype=np.float32,
)

bootstrap_rng = np.random.default_rng(
    MASTER_SEED
    + 960000
)

start_time = time.time()

for b in range(
    N_BOOTSTRAP
):
    y_star = simulate_parametric_bootstrap_y(
        bootstrap_rng,
        example["X"],
        example["pathogen_index"],
        example["K"],
        example_fit["beta_hat"],
        example_fit["sigma_g2_hat"],
        example_fit["sigma_e2_hat"],
    )

    fit_star = fit_section2_reml_gls(
        example["X"],
        y_star,
        example["pathogen_index"],
        example["K"],
        static=example_static,
    )

    _, delta_star = effects_from_beta(
        example["C"],
        example["P"],
        fit_star[
            "beta_hat"
        ],
    )

    bootstrap_well_delta[
        b,
        :,
    ] = delta_star[
        well_delta_mask
    ].astype(
        np.float32
    )

    bootstrap_poor_delta[
        b,
        :,
    ] = delta_star[
        poor_delta_mask
    ].astype(
        np.float32
    )

    bootstrap_well_dd[
        b,
        :,
    ] = (
        delta_star[
            well_i,
            well_j,
        ]
        - delta_star[
            well_k,
            well_j,
        ]
    ).astype(
        np.float32
    )

    bootstrap_poor_dd[
        b,
        :,
    ] = (
        delta_star[
            poor_i,
            poor_j,
        ]
        - delta_star[
            poor_k,
            poor_j,
        ]
    ).astype(
        np.float32
    )

    if (
        (b + 1) % 20 == 0
        or b == 0
        or b + 1 == N_BOOTSTRAP
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {b + 1:3d}/{N_BOOTSTRAP} bootstrap refits "
            f"({elapsed:.1f} seconds elapsed)"
        )


def summarize_bootstrap_group(
    label,
    bootstrap_values,
    true_values,
):
    low = np.quantile(
        bootstrap_values,
        0.025,
        axis=0,
    )

    high = np.quantile(
        bootstrap_values,
        0.975,
        axis=0,
    )

    contains_truth = (
        (low <= true_values)
        & (true_values <= high)
    )

    excludes_zero = (
        (low > 0)
        | (high < 0)
    )

    return {
        "effect":
            label,

        "number_of_effects":
            int(
                len(
                    true_values
                )
            ),

        "fraction_CI_excludes_zero":
            float(
                np.mean(
                    excludes_zero
                )
            ),

        "fraction_CI_contains_known_truth":
            float(
                np.mean(
                    contains_truth
                )
            ),

        "mean_CI_width":
            float(
                np.mean(
                    high
                    - low
                )
            ),
    }


bootstrap_summary = pd.DataFrame([
    summarize_bootstrap_group(
        "Delta_well_supported",
        bootstrap_well_delta,
        well_delta_true,
    ),
    summarize_bootstrap_group(
        "Delta_poorly_supported",
        bootstrap_poor_delta,
        poor_delta_true,
    ),
    summarize_bootstrap_group(
        "DeltaDelta_well_supported",
        bootstrap_well_dd,
        well_dd_true,
    ),
    summarize_bootstrap_group(
        "DeltaDelta_poorly_supported",
        bootstrap_poor_dd,
        poor_dd_true,
    ),
])

BOOTSTRAP_SUMMARY_PATH = (
    OUTPUT_DIR
    / "06_scenario6_support_bootstrap_summary.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

display(
    bootstrap_summary
)

print("\nSaved:")
print(
    BOOTSTRAP_SUMMARY_PATH
)

print("\nCell 10: PASS")


In [ ]:
#@title Cell 11 - Optional repeated-dataset bootstrap coverage

print(
    "Formal repeated-dataset bootstrap coverage remains optional and is "
    "disabled by default, as in the earlier simulations."
)

if RUN_FULL_BOOTSTRAP_COVERAGE:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=True was requested, but this notebook "
        "does not automatically launch the very expensive 100 x 200 coverage "
        "calculation. The representative support-group bootstrap in Cell 10 "
        "should be inspected first."
    )
else:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=False: no formal repeated-dataset "
        "coverage run was performed."
    )

print("\nCell 11: PASS")


In [ ]:
#@title Cell 12 - Final QC and output manifest

required_output_files = [
    MASK_PATH,
    SUPPORT_TABLE_PATH,
    REPLICATE_RESULTS_PATH,
    SUMMARY_PATH,
    BOOTSTRAP_SUMMARY_PATH,
]

missing_outputs = [
    str(
        path
    )
    for path in required_output_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Required Simulation 6 output(s) are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )

manifest = {
    "notebook":
        "12_Simulation_06_Weakly_Supported_Virtual_Combinations.ipynb",

    "scenario":
        "Simulation 06 - weakly supported virtual pathogen-plasmid combinations",

    "central_question":
        "Does recovery of Delta_ij and DeltaDelta_ikj become less accurate "
        "when the target virtual pathogen-plasmid combination is poorly "
        "represented by similar observed combinations?",

    "simulation1_biology_unchanged":
        True,

    "chromosome_support_features": {
        "number":
            D_C,

        "definition":
            "30 predefined AMR-related genes plus their 30 paired 300-bp putative promoter regions",

        "background_SNPs_in_support_distance":
            False,
    },

    "plasmid_support_features": {
        "number":
            D_P,

        "labels":
            PLASMID_FEATURE_LABELS,
    },

    "observation_pattern": {
        "P0_retained_for_all_pathogens":
            True,

        "cluster_centres":
            N_CLUSTER_CENTRES,

        "clustered_withheld_Pplus":
            N_CLUSTERED_WITHHELD,

        "random_withheld_Pplus":
            N_RANDOM_WITHHELD,

        "total_missing_Pplus_fraction":
            MISSING_PPLUS_FRACTION,

        "full_rank_observed_design_required":
            True,
    },

    "support_definition": {
        "nearest_observed_Pplus_combinations":
            N_SUPPORT_NEIGHBOURS,

        "chromosome_and_plasmid_blocks_equally_weighted":
            True,

        "well_supported":
            "lowest 25% of support distance among withheld targets",

        "poorly_supported":
            "highest 25% of support distance among withheld targets",
    },

    "delta_delta_comparison":
        "Each withheld target (i,j) is compared with pathogens k for which "
        "the same plasmid j is observed.",

    "primary_metrics": [
        "Delta RMSE: well versus poor support",
        "DeltaDelta RMSE: well versus poor support",
    ],

    "starting_meaningful_loss_ratio":
        GO_RMSE_RATIO,

    "n_simulation_replicates":
        N_SIM_REPLICATES,

    "n_representative_bootstrap_refits":
        N_BOOTSTRAP,

    "required_output_files": [
        str(
            path
        )
        for path in required_output_files
    ],
}

MANIFEST_PATH = (
    OUTPUT_DIR
    / "06_scenario6_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

print("=" * 90)
print("SIMULATION 6 NOTEBOOK COMPLETE")
print("=" * 90)

print(
    f"Manifest: "
    f"{MANIFEST_PATH}"
)

print(
    f"Required output files checked: "
    f"{len(required_output_files)}"
)

print(
    "\nOnly the pathogen-plasmid observation pattern/local support differs "
    "from the finalized Simulation 1 biological data-generating model."
)

print("Cell 12: PASS")
